## Quantitative evalution

In [1]:
# required modules (under Anaconda use: > conda install -c conda-forge <package>)
if False: # (skip if already installed)
    !pip install lark-parser
    !pip install linear-tree
    !pip install pydot
    !pip install pydotplus
    !pip install hopsy
    !pip install janus-swi
    !pip install dice-ml
    !pip install anchor-exp
    # download and install SWI Prolog from https://www.swi-prolog.org/download/stable

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# standard imports
import os
import sys
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text
from sklearn.metrics import accuracy_score

# black-box model imports
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

# dice
import dice_ml
# anchor
from anchor import utils
from anchor import anchor_tabular

# Using janus - set SWI Prolog home
SWI_HOME = r"C:\Program Files\swipl"
os.environ["SWI_HOME_DIR"] = SWI_HOME
os.add_dll_directory(os.path.join(SWI_HOME, "bin"))

# local imports
sys.path.append('../src/') # local path
import reasonx
import dautils
from helper_functions import read_adult, df_numeric_stats, local_neigh, read_give_me_some_credit, read_south_german_credit,\
    read_credit_card_default, read_australian_credit, get_l1_weights, get_linf_weights, get_distances

# for reproducibility
random.seed(42)
np.random.seed(42)

# Folder for saving plots
plots = 'plots/'

In [3]:
## move the settings for the section you want to reproduce at the bottom of this cell

# settings for section 'More on Quantitative Evaluation'
dataset = "adult"
depths = list(range(2,11))
norms = ["l1norm"]
dt_types = ['DT-M']
immutability = None # None = 1 random + (1+1) random
dice = False
more = True

# settings for section 'Quantitative Evaluation' - ReasonX only
depths = [3]
norms = ["l1norm", "linfnorm"]
dt_types = ['DT-M', 'DT-GS', 'DT-LS']
immutability = None 
dice = False
more = False
# also choose the dataset
#dataset = "aca"
#dataset = "ccd"
#dataset = "gmsc"
#dataset = "sgc"
dataset = "adult"

# settings for section 'Quantitative Evaluation' - comparison with ANCHOR
dataset = "adult"
depths = [3,4,5]
norms = ["l1norm"]
dt_types = ['DT-GS']
probs = [.8, .9, .95]
dice = False
anchortest = True
more = False

# settings for section 'Quantitative Evaluation' - comparison with DICE
dataset = "adult"
depths = [3,4,5]
norms = ["l1norm", "linfnorm"]
dt_types = ['DT-GS']
immutability = ('hoursperweek', ('race', 'sex')) # for comparison
n_total_ce = [2, 3, 4, 5]
dice = True
anchortest = False
more = False


In [4]:
# read dataset
simplified=False
continuous_only=False
if dataset == "gmsc":
    df, pred_atts, target, df_code = read_give_me_some_credit(continuous_only=continuous_only, simplified=simplified)
if dataset == "sgc":
    df, pred_atts, target, df_code = read_south_german_credit(continuous_only=continuous_only, simplified=simplified)
if dataset == "adult":
    df, pred_atts, target, df_code = read_adult(continuous_only=continuous_only, simplified=simplified)
if dataset == "ccd":
    df, pred_atts, target, df_code = read_credit_card_default(continuous_only=continuous_only, simplified=simplified)
if dataset == "aca":
    df, pred_atts, target, df_code = read_australian_credit(continuous_only=continuous_only, simplified=simplified)

# numeric_stats used in local neigh generation
numeric_stats = df_numeric_stats(df, pred_atts, df_code)
print(numeric_stats)
df.info()

{'capitalloss': (403.004552124359, 0, 4356, 0, dtype('int64')), 'capitalgain': (7452.019057655394, 0, 99999, 0, dtype('int64')), 'hoursperweek': (12.391444024252307, 1, 99, 0, dtype('int64')), 'age': (13.710509934443557, 17, 90, 0, dtype('int64'))}
<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   race          48842 non-null  str  
 1   sex           48842 non-null  str  
 2   workclass     48842 non-null  str  
 3   education     48842 non-null  str  
 4   age           48842 non-null  int64
 5   capitalgain   48842 non-null  int64
 6   capitalloss   48842 non-null  int64
 7   hoursperweek  48842 non-null  int64
 8   class         48842 non-null  str  
dtypes: int64(4), str(5)
memory usage: 3.4 MB


In [5]:
# Encoding the dataset
df_encoded_onehot = df_code.fit_transform(df)
encoded_pred_atts = df_code.encoded_atts(pred_atts)
l1weights = get_l1_weights(encoded_pred_atts, pred_atts, df_code)
linfweights = get_linf_weights(encoded_pred_atts, pred_atts, df_code)
#df_encoded_onehot.head()

In [6]:
# split predictive and target
X, y = df_encoded_onehot[encoded_pred_atts], df_encoded_onehot[target]

# retain test sets
X1, XT1, y1, yt1 = train_test_split(X, y, test_size=0.3, random_state=42)

# train different models
mlp = MLPClassifier(random_state=0)
mlp.fit(X1, y1)

rf = RandomForestClassifier(max_depth=3, n_estimators=100, random_state=42)
rf.fit(X1, y1)

xgb = XGBClassifier(random_state=42)
xgb.fit(X1, y1)

# simple performance comparison of models
print("MLP              ", mlp.score(XT1, yt1))
print("RF               ", rf.score(XT1, yt1))
print("XGB              ", xgb.score(XT1, yt1))

MLP               0.8156009008394185
RF                0.8037261994130894
XGB               0.851361495939398


In [7]:
# choose black box
#bb = mlp
#bb = rf
bb = xgb

In [8]:
bb_label = bb.predict(XT1)

# split the test set (XT1/xgb_labels) in two parts
XT1_train, XT1_test, bb_label_train, bb_label_test = train_test_split(XT1, bb_label, test_size=0.3, random_state=42)
#print(len(XT1_train), len(XT1_test))

In [9]:
# useful for DICE
import time

class BB_DICE:
    def __init__(self, bb_model, transform):
        self.bb = bb_model
        self.transform = transform
        self.time = 0 
        if hasattr(bb_model, "classes_"):
            self.classes_ = bb_model.classes_

    def predict(self, df):
        start = time.perf_counter()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)            
        df_trasformed = self.transform(df)
        self.time += time.perf_counter() - start
        return self.bb.predict(df_trasformed)

    def predict_proba(self, df):
        start = time.perf_counter()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)            
        df_trasformed = self.transform(df)
        self.time += time.perf_counter() - start
        return self.bb.predict_proba(df_trasformed)

def dice_cf(x_features, immutable, total_CF, df_code):
    features_to_vary = sorted(list( set(pred_atts) - set(immutable) ))
    dice_x_features = dice_test.loc[x_features.index]
    dice_bb.time = 0 # reset df transformation time
    start = time.perf_counter()
    try:
        # DiCE allows tunable parameters proximity_weight (default: 0.5) and diversity_weight (default: 1.0) to handle proximity and diversity respectively.
        e1 = dice_exp.generate_counterfactuals(dice_x_features, total_CFs=total_CF, desired_class="opposite", 
                                               features_to_vary=features_to_vary, proximity_weight=0.5, diversity_weight=0)
        res = e1.cf_examples_list[0].final_cfs_df
    except Exception:
        res = dice_test.iloc[0:0].copy()    
    eps = time.perf_counter() - start - dice_bb.time # subtract time for df transformations
    res_encoded = df_code.transform(res)
    return res_encoded, eps

if dice:
    dice_train = df.loc[X1.index].copy()
    dice_train[target] = y1
    dice_test = df.loc[XT1.index].copy()
    dice_test = dice_test[pred_atts]
    #dice_test[target] = 0 # check?
    continuous_atts = list(set(pred_atts)-set(df_code.nominal)-set(df_code.ordinal))
    dice_data  = dice_ml.Data(dataframe=dice_train, continuous_features=continuous_atts, outcome_name=target)
    dice_bb = BB_DICE(bb, df_code.transform)
    dice_m = dice_ml.Model(model=dice_bb, backend="sklearn")
    dice_exp = dice_ml.Dice(dice_data, dice_m, method="genetic")

# temp - for testing
if dice and False:
    i=0
    x_features = XT1.iloc[i:(i+1)]
    #print(x_features)
    res, tm = dice_cf(x_features, immutability[0], 2, df_code)
    dists = get_distances(x_features, res.iloc[:,:-1], l1weights, linfweights)
    print(dists)
    print(tm, dice_bb.time)


In [10]:
# useful for ANCHOR
import copy

# transform from one hot encoded to format for anchor
def adult_transform(x_features):
    to_anchor = df_code_no1hot.transform(df_code.inverse_transform(x_features))
    to_anchor['education'] = to_anchor['education']-1
    return to_anchor.to_numpy()

# transform from anchor format to one hot encoded for bb
def adult_inverse_transform(x_features):
    x_features['education'] = x_features['education']+1
    x_features = df_code.transform( df_code_no1hot.inverse_transform(x_features) )
    return x_features    

class BB_ANCHOR:
    def __init__(self, bb_model, transform, df_format):
        self.bb = bb_model
        self.transform = transform
        self.df_format_columns = df_format.columns
        self.df_format_dtypes = df_format.dtypes
        self.time = 0 
        if hasattr(bb_model, "classes_"):
            self.classes_ = bb_model.classes_

    def predict(self, df):
        start = time.perf_counter()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df, columns=self.df_format_columns).astype(self.df_format_dtypes)           
        df_trasformed = self.transform(df)
        self.time += time.perf_counter() - start
        return self.bb.predict(df_trasformed)

    def predict_proba(self, df):
        start = time.perf_counter()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df, columns=self.df_format_columns).astype(self.df_format_dtypes)           
        df_trasformed = self.transform(df)
        self.time += time.perf_counter() - start
        return self.bb.predict_proba(df_trasformed)
    
if anchortest:
    # df without one-hot encoding
    df_code_no1hot = copy.deepcopy(df_code)
    df_code_no1hot.onehot = False
    df_no1hot = df_code_no1hot.fit_transform(df)
    df_no1hot = df_no1hot[pred_atts]
    # initialize the explanator on encoded data
    categorical_atts = list((df_code.nominal | df_code.ordinal) - set([target]))
    df_no1hot[categorical_atts] = df_no1hot[categorical_atts].astype(int)
    categorical_names = { pred_atts.index(f): [df_code.decode[f][k] for k in sorted(df_code.decode[f])] for f in categorical_atts }
    class_names = [df_code.decode[target][k] for k in sorted(df_code.decode[target])] 
    X1_anchor = adult_transform(X1)
    anchor_exp = anchor_tabular.AnchorTabularExplainer(
        class_names = class_names,
        feature_names = pred_atts,
        train_data = X1_anchor,
        categorical_names = categorical_names)
    bb_anchor = BB_ANCHOR(bb, adult_inverse_transform, df_no1hot)

# temp - for testing
if anchortest and False:
    i=0
    x_features = XT1.iloc[i:(i+1)]
    #print(x_features)
    anchor_x = adult_transform(x_features)
    #print(anchor_x)
    #print(bb_anchor.predict(anchor_x))
    np.random.seed(42)
    random.seed(42)
    exp = anchor_exp.explain_instance(anchor_x, bb_anchor.predict, threshold=.95)
    print('Anchor: %s' % (','.join(exp.names())))

### Experimenting

In [11]:
# experiment data collection
def stats(r):
    r.solveopt()
    factual = r.rules('F', distinct=True)
    # Factual rule
    if len(factual)>0:
        frule, _, _ = factual[0]
        lfrule = len(frule.split(','))
    else:
        lfrule = None
    start = time.perf_counter()
    r.solveopt(minimize=norm+'(F, CE)', project=['CE'], eps=0.01)
    eps = time.perf_counter() - start
    totd = r.minvalues()
    # dimensionality    
    ans = r.answers()
    nans = len(ans)
    maxdim = None
    if nans!=0:
        # find largest dimensionality
        for an in ans:
            try:
                text, var_to_pos, var_to_const = r.hopsy_parse(an)
                dim = len(pred_atts) - len(var_to_const)
                if maxdim is None or dim > maxdim:
                    maxdim = dim
            except ValueError:
                continue
        #if len(var_to_const) == len(pred_atts):
        #    print(text, var_to_pos, var_to_const)
    # CE
    r.solveopt(project=['CE'])
    ans = r.answers()
    nans = len(ans)
    totnc = sum( c.count(",") for c in ans ) + nans
    #print(eps, nans, totnc, totd)
    return eps, nans, totnc, totd, lfrule, maxdim

def stats_factual(r):
    start = time.perf_counter()
    r.solveopt()
    eps = time.perf_counter() - start
    factual = r.rules('F', distinct=True)
    # Factual rule
    if len(factual)>0:
        frule, fprob, fcov = factual[0]
        lfrule = len(frule.split(','))
    else:
        lfrule, fprob, fcov = None, None, None
    return eps, lfrule, fprob, fcov

In [12]:
%%time
# train DTs

instances = list(range(0, 100))
ninstances = len(instances)
clfs, fids = dict(), dict()

for depth in depths:
    for id in ['DT-M', 'DT-GS','DT-LS']:
        clfs[(id,depth)] = []
        fids[(id,depth)] = []
        clf = DecisionTreeClassifier(max_depth=depth)
        match id:
            case 'DT-M':
                clf.fit(X1, y1)
                clfs[(id,depth)] = clf
                fids[(id,depth)] = clf.score(XT1, yt1)
            case 'DT-GS':
                clf.fit(XT1_train, bb_label_train)
                clfs[(id,depth)] = clf
                fids[(id,depth)] = clf.score(XT1_test, bb_label_test)
            case 'DT-LS':
                for i in instances:
                    x_features = XT1.iloc[i:(i+1)]
                    #x_class = bb.predict(x_features)[0] # not needed
                    df_neigh = local_neigh(x_features, encoded_pred_atts, numeric_stats, N=10000)
                    clf = DecisionTreeClassifier(max_depth=depth)
                    neigh_bb_label = bb.predict(df_neigh)
                    # split the neighboorhood
                    df_neigh_train, df_neigh_test, neigh_bb_label_label_train, neigh_bb_label_label_test = train_test_split(df_neigh, neigh_bb_label, test_size=0.3, random_state=42)
                    clf = DecisionTreeClassifier(max_depth=depth)
                    clf.fit(df_neigh_train, neigh_bb_label_label_train)
                    clfs[(id,depth)].append(clf)
                    fids[(id,depth)].append(clf.score(df_neigh_test, neigh_bb_label_label_test))

CPU times: total: 12min
Wall time: 23.8 s


In [13]:
%%time
# main loop

times, nanswers, lanswers, distances  = dict(), dict(), dict(), dict()
f_length, dimens, dice_dist, dice_time = dict(), dict(), dict(), dict()
anchor_info, anchor_time = dict(), dict()

for depth in depths:
    for id in dt_types:      
        for norm in norms:
            nr0, nr1, nr2 = dict(), dict(), dict()
            lr0, lr1, lr2 = dict(), dict(), dict()
            lfr0, lfr1, lfr2 = dict(), dict(), dict()
            dr0, dr1, dr2 = dict(), dict(), dict()
            times0, times1, times2 = dict(), dict(), dict()
            dm0, dm1, dm2 = dict(), dict(), dict()
            diced0, diced1, diced2 = dict(), dict(), dict()
            dicet0, dicet1, dicet2 = dict(), dict(), dict()
            for i in instances:
                if id=='DT-GS' and ((dataset=='sgc' and i==90) or (dataset=="aca" and i==63)): # not enough instances
                    break
                if i % 25 == 0:
                    print(id, depth, norm, i)
                r = reasonx.ReasonX(pred_atts, target, df_code, verbose=0)
                # retrieve DT
                match id:
                    case 'DT-M':
                        r.model(clfs[(id,depth)])
                    case 'DT-GS':
                        r.model(clfs[(id,depth)], bb=bb)
                    case 'DT-LS':
                        r.model(clfs[(id,depth)][i], bb=bb)
                x_features = XT1.iloc[i:(i+1)] if id!='DT-GS' else XT1_test.iloc[i:(i+1)]
                x_class = yt1.iloc[i] if id=='DT-M' else bb.predict(x_features)[0]
                if anchortest:
                    for prob in probs:
                        start = time.perf_counter()
                        r.instance('F', features=x_features, label=x_class, minconf=prob)  
                        eps0 = time.perf_counter() - start
                        times0[(prob,i)], lfr0[(prob,i)], dr0[(prob,i)], dr1[(prob,i)] = stats_factual(r)  
                        times0[(prob,i)] += eps0                      
                else:
                    start = time.perf_counter()
                    r.instance('F', features=x_features, label=x_class)  
                    r.instance('CE', label=1-x_class)   
                    eps0 = time.perf_counter() - start
                    times0[i], nr0[i], lr0[i], dr0[i], lfr0[i], dm0[i] = stats(r)  
                    times0[i] += eps0
                    random_f = random.choice(pred_atts) if immutability is None else immutability[0]
                    start = time.perf_counter()
                    r.constraint(f'F.{random_f}=CE.{random_f}')
                    eps1 = time.perf_counter() - start
                    times1[i], nr1[i], lr1[i], dr1[i], lfr1[i], dm1[i] = stats(r)
                    times1[i] += eps0 + eps1
                    if immutability is None:
                        random_f2 = random.choice([f for f in pred_atts if f!=random_f])
                        start = time.perf_counter()
                        r.constraint(f'F.{random_f2}=CE.{random_f2}')
                        eps2 = time.perf_counter() - start
                    else:
                        r.retract(last=True)
                        f1, f2 = immutability[1]
                        start = time.perf_counter()
                        r.constraint(f'F.{f1}=CE.{f1},F.{f2}=CE.{f2}')
                        eps2 = (time.perf_counter() - start) - eps1 # do not count eps1
                    times2[i], nr2[i], lr2[i], dr2[i], lfr2[i], dm2[i] = stats(r)
                    times2[i] += eps0 + eps1 + eps2
                if dice and id == 'DT-GS' and norm==norms[0] and depth==depths[0]: # dice is independent from the type, depth, norm
                    #print('instance', i)
                    dice_dist1, dice_dist2 = dict(), dict()
                    for nCE in n_total_ce:
                        res, eps = dice_cf(x_features, [], nCE, df_code)
                        diced0[(nCE,i)], dicet0[(nCE,i)] = res, eps    
                        #print(nCE, 'res0', len(res))   
                        res, eps = dice_cf(x_features, [random_f], nCE, df_code)
                        #dists1 = dice_dist(x_features, res, norm=norm)
                        diced1[(nCE,i)], dicet1[(nCE,i)] = res, eps    
                        #print(nCE, 'res1', len(res))   
                        res, eps = dice_cf(x_features, [random_f, random_f2] if immutability is None else [f1, f2], nCE, df_code)
                        #dists2 = dice_dist(x_features, res, norm=norm)
                        diced2[(nCE,i)], dicet2[(nCE,i)] = res, eps
                        #print(nCE, 'res2', len(res))
                if anchortest and id == 'DT-GS' and norm==norms[0] and depth==depths[0]: # anchortest is independent from the type, depth, norm
                    #print('instance', i)
                    for prob in probs:
                        #print('conf', prob)
                        anchor_x = adult_transform(x_features)
                        np.random.seed(42)
                        random.seed(42)
                        bb_anchor.time = 0
                        start = time.perf_counter()
                        exp = anchor_exp.explain_instance(anchor_x, bb_anchor.predict, threshold=prob)
                        eps = time.perf_counter() - start
                        anchor_info[(prob,i)] = (len(exp.names()), exp.precision(), exp.coverage())
                        anchor_time[(prob,i)] = eps - bb_anchor.time # remove transformation time
    
            times[(id,depth,norm,0)], times[(id,depth,norm,1)], times[(id,depth,norm,2)] = times0, times1, times2
            nanswers[(id,depth,norm,0)], nanswers[(id,depth,norm,1)], nanswers[(id,depth,norm,2)] = nr0, nr1, nr2
            lanswers[(id,depth,norm,0)], lanswers[(id,depth,norm,1)], lanswers[(id,depth,norm,2)] = lr0, lr1, lr2
            distances[(id,depth,norm,0)], distances[(id,depth,norm,1)], distances[(id,depth,norm,2)] = dr0, dr1, dr2  
            f_length[(id,depth,norm,0)], f_length[(id,depth,norm,1)], f_length[(id,depth,norm,2)] = lfr0, lfr1, lfr2 
            dimens[(id,depth,norm,0)], dimens[(id,depth,norm,1)], dimens[(id,depth,norm,2)] = dm0, dm1, dm2
            dice_dist[(id,depth,norm,0)], dice_dist[(id,depth,norm,1)], dice_dist[(id,depth,norm,2)] = diced0, diced1, diced2
            dice_time[(id,depth,norm,0)], dice_time[(id,depth,norm,1)], dice_time[(id,depth,norm,2)] = dicet0, dicet1, dicet2

DT-GS 3 l1norm 0


100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


DT-GS 3 l1norm 25


100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


DT-GS 3 l1norm 50


100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


DT-GS 3 l1norm 75


100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


DT-GS 3 linfnorm 0
DT-GS 3 linfnorm 25
DT-GS 3 linfnorm 50
DT-GS 3 linfnorm 75
DT-GS 4 l1norm 0
DT-GS 4 l1norm 25
DT-GS 4 l1norm 50
DT-GS 4 l1norm 75
DT-GS 4 linfnorm 0
DT-GS 4 linfnorm 25
DT-GS 4 linfnorm 50
DT-GS 4 linfnorm 75
DT-GS 5 l1norm 0
DT-GS 5 l1norm 25
DT-GS 5 l1norm 50
DT-GS 5 l1norm 75
DT-GS 5 linfnorm 0
DT-GS 5 linfnorm 25
DT-GS 5 linfnorm 50
DT-GS 5 linfnorm 75
CPU times: total: 15h 20min 51s
Wall time: 42min 34s


In [14]:
# ANCHOR Table
if anchortest:
    norm = norms[0]
    for prob in probs:
        rlength = [anchor_info[(prob,i)][0] for i in instances if anchor_info[(prob,i)] is not None]
        length = np.mean(rlength)
        prec = np.mean([anchor_info[(prob,i)][1] for i in instances if anchor_info[(prob,i)] is not None])
        cov = np.mean([anchor_info[(prob,i)][2] for i in instances if anchor_info[(prob,i)] is not None])
        elapsed = np.mean([anchor_time[(prob,i)] for i in instances if anchor_time[(prob,i)]])
        sn = len(rlength)/len(instances)
        # & $D$ & $f$ & $\mathit{MC}_F$ & $S/N$ & \textbf{$l_F$} & \textbf{$p$} & $c$ & time 
        print(f'ANCHOR & & & {prob:.2f} & {sn:.2f} & {length:.2f} & {prec:.3f} & {cov:.3f} & {elapsed:.3f} \\\\')
    print(r'\midrule')
    for id in ['DT-GS']:
        #print(dataset, 'ncon', ncon, 'depth', depth)
        for d in depths:
            for prob in probs:
                fid = np.mean(fids[(id,d)]) if id=='DT-LS' else fids[(id,d)] 
                rlength = [l for (prob1, _), l in f_length[(id,d,norm,0)].items() if prob1==prob and l is not None]
                length = np.mean(rlength)
                sn = len(rlength)/len(instances)
                prec = np.mean( [p for (prob1, _), p in distances[(id,d,norm,0)].items() if prob1==prob and p is not None])
                cov = np.mean( [p for (prob1, _), p in distances[(id,d,norm,1)].items() if prob1==prob and p is not None])
                elapsed = np.mean([t for (prob1, _), t in times[(id,d,norm,0)].items() if prob1==prob and t is not None])
                # & $D$ & $f$ & $\mathit{MC}_F$ & $S/N$ & \textbf{$l_F$} & \textbf{$p$} & $c$ & time 
                print(f'REASONX & {d} & {fid:.3f} & {prob:.2f} & {sn:.2f} & {length:.2f} & {prec:.3f} &  {cov:.3f} & {elapsed:.3f} \\\\')


In [15]:
# DICE Table
if dice:
    norm = norms[0]
    for ncon in [0, 1, 2]:
        print(r'%', dataset, 'constraint', '-' if ncon==0 else immutability[ncon-1])
        for nce in n_total_ce:
            diced = dice_dist[('DT-GS',depths[0],'l1norm',ncon)]
            mindist1norm, mindistinf = [], []
            ns = 0
            for i in instances:
                x_features = XT1_test.iloc[i:(i+1)]
                res = diced[(nce, i)]
                if len(res)==0:
                    ns += 1
                    continue
                dist1, distinf = get_distances(x_features,res.iloc[:,:-1], l1weights, linfweights)
                mindist1norm.append(np.min(dist1))
                mindistinf.append(np.min(distinf))
            sn = 1-ns/len(instances)
            d1 = np.mean(mindist1norm)
            d2 = np.mean(mindistinf)
            elapsed = np.mean([t for (ce, _), t in dice_time[('DT-GS',depths[0],'l1norm',ncon)].items() if ce==nce])
            print(f'DiCE & & & {sn:.2f} & {nce} & {d1:.3f} & {d2:.3f} & \\multicolumn{{2}}{{c}}{{{elapsed:.3f}}} \\\\')
        print(r'%')
        for id in dt_types:
            #print(dataset, 'ncon', ncon, 'depth', depth)
            for d in depths:
                fid = np.mean(fids[(id,d)]) if id=='DT-LS' else fids[(id,d)] 
                rlength = [l for l in f_length[(id,depth,norm,ncon)].values() if l is not None]
                sn = len(rlength)/len(f_length[(id,depth,norm,ncon)])
                nce = np.mean([n for n in nanswers[(id,d,norm,ncon)].values()])
                d1 = np.mean( [np.min(ds) for ds in distances[(id,d,'l1norm',ncon)].values() if len(ds)>0] )
                d2 = np.mean( [np.min(ds) for ds in distances[(id,d,'linfnorm',ncon)].values() if len(ds)>0] )
                elapsed1 = np.mean([n for n in times[(id,d,'l1norm',ncon)].values()])
                elapsed2 = np.mean([n for n in times[(id,d,'linfnorm',ncon)].values()])
                print(f'REASONX & {d} & {fid:.3f} & {sn:.2f} & {nce:.2f} & {d1:.3f} & {d2:.3f} & {elapsed1:.3f} & {elapsed2:.3f} \\\\')                


% adult constraint -
DiCE & & & 1.00 & 2 & 1.036 & 0.687 & \multicolumn{2}{c}{0.389} \\
DiCE & & & 1.00 & 3 & 0.896 & 0.640 & \multicolumn{2}{c}{0.584} \\
DiCE & & & 1.00 & 4 & 0.855 & 0.621 & \multicolumn{2}{c}{0.689} \\
DiCE & & & 1.00 & 5 & 0.821 & 0.602 & \multicolumn{2}{c}{0.597} \\
%
REASONX & 3 & 0.921 & 0.96 & 4.34 & 0.065 & 0.060 & 0.078 & 0.168 \\
REASONX & 4 & 0.940 & 0.96 & 7.44 & 0.047 & 0.048 & 0.133 & 0.321 \\
REASONX & 5 & 0.948 & 0.96 & 14.40 & 0.047 & 0.044 & 0.286 & 0.761 \\
% adult constraint hoursperweek
DiCE & & & 1.00 & 2 & 0.998 & 0.681 & \multicolumn{2}{c}{0.662} \\
DiCE & & & 1.00 & 3 & 0.896 & 0.626 & \multicolumn{2}{c}{0.984} \\
DiCE & & & 1.00 & 4 & 0.832 & 0.606 & \multicolumn{2}{c}{1.041} \\
DiCE & & & 1.00 & 5 & 0.846 & 0.636 & \multicolumn{2}{c}{1.979} \\
%
REASONX & 3 & 0.921 & 0.96 & 4.34 & 0.065 & 0.060 & 0.064 & 0.158 \\
REASONX & 4 & 0.940 & 0.96 & 6.23 & 0.061 & 0.058 & 0.105 & 0.332 \\
REASONX & 5 & 0.948 & 0.96 & 12.29 & 0.049 & 0.047 & 0.262 & 

In [16]:
# REASONX-only Table (repeat for each dataset)
if not more and not dice and not anchortest:
    ncon = 0
    d, norm = 3, norms[0]
    print(dataset)
    for id in dt_types:
        fid = np.mean(fids[(id,d)]) if id=='DT-LS' else fids[(id,d)] 
        rlength = [l for l in f_length[(id,depth,norm,ncon)].values() if l is not None]
        lfrule = np.mean(rlength)
        sn = len(rlength)/len(f_length[(id,depth,norm,ncon)])
        nce = np.mean([n for n in nanswers[(id,d,norm,ncon)].values()])
        nansw = np.sum([n for n in nanswers[(id,d,norm,ncon)].values()])
        lce = np.sum([n for n in lanswers[(id,d,norm,ncon)].values()])/nansw
        d1 = np.sum([n for ds in distances[(id,d,'l1norm',ncon)].values() for n in ds] )/nansw
        d2 = np.sum([n for ds in distances[(id,d,'linfnorm',ncon)].values() for n in ds])/nansw
        dim1 = max([n for n in dimens[(id,d,'l1norm',ncon)].values() if n is not None])
        dim2 = max([n for n in dimens[(id,d,'linfnorm',ncon)].values() if n is not None])
        print(f'& {id} & {fid:.3f} & {sn:.2f} & {lfrule:.2f} & {nce:.2f} & {lce:.2f} & {d1:.3f} & {d2:.3f} & {dim1} & {dim2}\\\\')


In [17]:
# the rest of the notebook is for supplemental material section 'More on Quantitative Evaluation'

In [18]:
import matplotlib.pyplot as plt

# general settings  
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc('font', size=14)
plt.rc('legend', fontsize=14)
plt.rc('lines', linewidth=2)
plt.rc('axes', linewidth=2)
plt.rc('axes', edgecolor='k')
plt.rc('xtick.major', width=2)
plt.rc('xtick.major', size=6)
plt.rc('ytick.major', width=2)
plt.rc('ytick.major', size=6)
plt.rc('pdf', fonttype=42)
plt.rc('ps', fonttype=42)

In [19]:
def plot_results_info(norm='l1norm'):
    plt.figure(figsize=(12, 4))
    plt.subplots_adjust(hspace=0.4)
    
    s_mean = [np.mean([clf.get_n_leaves() for clf in clfs[('DT-LS',d)]]) for d in depths]
    s_std = [np.std([clf.get_n_leaves() for clf in clfs[('DT-LS',d)]]) for d in depths]

    plt.subplot(1, 2, 1)
    plt.plot(depths, [clfs[('DT-M',d)].get_n_leaves() for d in depths], marker = 'o', alpha = 0.5, color = 'tomato', label = "DT-M")
    plt.plot(depths, [clfs[('DT-GS',d)].get_n_leaves() for d in depths], marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "DT-GS")
    plt.errorbar(depths, s_mean, s_std, marker = 'o', alpha = 0.5, color = 'darkblue', label = "DT-LS")
    plt.title("no. leaves")
    plt.legend()
    plt.xlabel("base model depth $D$")

    f_mean = [np.mean([fid for fid in fids[('DT-LS',d)]]) for d in depths]
    f_std = [np.std([fid for fid in fids[('DT-LS',d)]]) for d in depths]

    plt.subplot(1, 2, 2)
    plt.plot(depths, [fids[('DT-M',d)] for d in depths], marker = 'o', alpha = 0.5, color = 'tomato', label = "DT-M")
    plt.plot(depths, [fids[('DT-GS',d)] for d in depths], marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "DT-GS")
    plt.errorbar(depths, f_mean, f_std, marker = 'o', alpha = 0.5, color = 'darkblue', label = "DT-LS")
    plt.title("accuracy/fidelity")
    plt.ylim([.75, 1.02])
    plt.legend()
    plt.xlabel("base model depth $D$")

if more:
    plot_results_info()
    plt.savefig(plots+'further1.pdf', bbox_inches='tight', dpi=400)
    plt.show()

In [20]:
def plot_results(id):
    plt.figure(figsize=(12, 9))
    plt.subplots_adjust(hspace=0.4)
    
    r0_mean = [np.mean([n for n in nanswers[(id,d,norm,0)].values()]) for d in depths]
    r0_std = [np.std([n for n in nanswers[(id,d,norm,0)].values()]) for d in depths]
    r1_mean = [np.mean([n for n in nanswers[(id,d,norm,1)].values()]) for d in depths]
    r1_std = [np.std([n for n in nanswers[(id,d,norm,1)].values()]) for d in depths]
    r2_mean = [np.mean([n for n in nanswers[(id,d,norm,2)].values()]) for d in depths]
    r2_std = [np.std([n for n in nanswers[(id,d,norm,2)].values()]) for d in depths]
    
    plt.subplot(2,2,1)
    plt.errorbar(depths, r0_mean, r0_std, marker = 'o', alpha = 0.5, color = 'tomato', label = "0 eq. constraints")
    plt.errorbar(depths, r1_mean, r1_std, marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "1 eq. constraints")
    plt.errorbar(depths, r2_mean, r2_std, marker = 'o', alpha = 0.5, color = 'darkblue', label = "2 eq. constraints")
    plt.title("number of CE")
    plt.legend()
    plt.xlabel("base model depth $D$")
    plt.ylabel("$N_{CE}$")
    
    t0_mean = [np.mean([n for n in times[(id,d,norm,0)].values()]) for d in depths]
    t0_std = [np.std([n for n in times[(id,d,norm,0)].values()]) for d in depths]
    t1_mean = [np.mean([n for n in times[(id,d,norm,1)].values()]) for d in depths]
    t1_std = [np.std([n for n in times[(id,d,norm,1)].values()]) for d in depths]
    t2_mean = [np.mean([n for n in times[(id,d,norm,2)].values()]) for d in depths]
    t2_std = [np.std([n for n in times[(id,d,norm,2)].values()]) for d in depths]

    plt.subplot(2,2,2)
    plt.errorbar(depths, t0_mean, t0_std, marker = 'o', alpha = 0.5, color = 'tomato', label = "0 eq. constraints")
    plt.errorbar(depths, t1_mean, t1_std, marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "1 eq. constraints")
    plt.errorbar(depths, t2_mean, t2_std, marker = 'o', alpha = 0.5, color = 'darkblue', label = "2 eq. constraints")
    plt.title("elapsed time")
    plt.legend()
    plt.xlabel("base model depth $D$")
    plt.ylabel("time (s)")

    d0_mean = [np.sum([n for ds in distances[(id,d,norm,0)].values() for n in ds])/
               np.sum([n for n in nanswers[(id,d,norm,0)].values()]) for d in depths]
    d1_mean = [np.sum([n for ds in distances[(id,d,norm,1)].values() for n in ds])/
               np.sum([n for n in nanswers[(id,d,norm,1)].values()]) for d in depths]
    d2_mean = [np.sum([n for ds in distances[(id,d,norm,2)].values() for n in ds])/
               np.sum([n for n in nanswers[(id,d,norm,2)].values()]) for d in depths]
     
    plt.subplot(2,2,4)
    plt.plot(depths, d0_mean, marker = 'o', alpha = 0.5, color = 'tomato', label = "0 eq. constraints")
    plt.plot(depths, d1_mean, marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "1 eq. constraints")
    plt.plot(depths, d2_mean, marker = 'o', alpha = 0.5, color = 'darkblue', label = "2 eq. constraints")
    #plt.errorbar(depths, d0_mean, r0_std, marker = 'o', alpha = 0.5, color = 'tomato', label = "admissible")
    #plt.errorbar(depths, d1_mean, r1_std, marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "1 eq. constraints")
    #plt.errorbar(depths, d2_mean, r2_std, marker = 'o', alpha = 0.5, color = 'darkblue', label = "2 eq. constraints")
    plt.title("L$_1$ distance of MCE")
    plt.legend()
    plt.xlabel("base model depth $D$")
    plt.ylabel("$d_{MCE}(L_1)$")

    l0_mean = [np.sum([n for n in lanswers[(id,d,norm,0)].values()])/np.sum([n for n in nanswers[(id,d,norm,0)].values()]) for d in depths]
    l1_mean = [np.sum([n for n in lanswers[(id,d,norm,1)].values()])/np.sum([n for n in nanswers[(id,d,norm,1)].values()]) for d in depths]
    l2_mean = [np.sum([n for n in lanswers[(id,d,norm,2)].values()])/np.sum([n for n in nanswers[(id,d,norm,2)].values()]) for d in depths]
     
    plt.subplot(2,2,3)
    plt.plot(depths, l0_mean, marker = 'o', alpha = 0.5, color = 'tomato', label = "0 eq. constraints")
    plt.plot(depths, l1_mean, marker = 'o', alpha = 0.5, color = 'cornflowerblue', label = "1 eq. constraints")
    plt.plot(depths, l2_mean, marker = 'o', alpha = 0.5, color = 'darkblue', label = "2 eq. constraints")
    plt.title("CE rule length")
    plt.legend()
    plt.xlabel("base model depth $D$")
    plt.ylabel("$l_{CE}$")

if more:
    plot_results('DT-M')
    plt.savefig(plots+'further2.pdf', bbox_inches='tight', dpi=400)
    plt.show()